### CORE IDEA EXPANSION

#### ROTATING THE BASE STATE IN ORACE TO FIND |+++> INSTEAD OF |111> TO CHECK IF HAMMING WEIGHTS STILL WORK UNDER NON_COMPUTATIONAL BASIS STATES

In [1]:
from qiskit import QuantumCircuit
from qiskit_aer.primitives import Sampler

In [16]:
def oracle_plus_plus_plus(qc):
    """Oracle that marks |+++⟩"""
    qc.h([0, 1, 2])  # Convert to X-basis
    # Mark |000⟩ in X-basis (which is |+++⟩ in Z-basis)
    qc.x([0, 1, 2])
    qc.h(2)
    qc.ccx(0, 1, 2)
    qc.h(2)
    qc.x([0, 1, 2])
    qc.h([0, 1, 2])
    
def diffuser(qc):
    # Standard diffuser
    qc.h([0,1,2])
    qc.x([0,1,2])
    qc.h(2)
    qc.ccx(0, 1, 2)
    qc.h(2)
    qc.x([0,1,2])
    qc.h([0,1,2])

qc = QuantumCircuit(3, 3)

# Initialize uniform superposition
qc.h([0,1,2])

# Number of Grover iterations = 2
for _ in range(2):
    oracle_plus_plus_plus(qc)
    diffuser(qc)

# Measurement
qc.measure([0,1,2], [0,1,2])

qc.draw()

┌───┐┌───┐┌───┐          ┌───┐┌───┐┌───┐┌───┐               ┌───┐┌───┐»
q_0: ┤ H ├┤ H ├┤ X ├───────■──┤ X ├┤ H ├┤ H ├┤ X ├────────────■──┤ X ├┤ H ├»
     ├───┤├───┤├───┤       │  ├───┤├───┤├───┤├───┤            │  ├───┤├───┤»
q_1: ┤ H ├┤ H ├┤ X ├───────■──┤ X ├┤ H ├┤ H ├┤ X ├────────────■──┤ X ├┤ H ├»
     ├───┤├───┤├───┤┌───┐┌─┴─┐├───┤├───┤├───┤├───┤┌───┐┌───┐┌─┴─┐├───┤├───┤»
q_2: ┤ H ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├»
     └───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘»
c: 3/══════════════════════════════════════════════════════════════════════»
                                                                           »
«     ┌───┐┌───┐               ┌───┐┌───┐┌───┐┌───┐               ┌───┐┌───┐»
«q_0: ┤ H ├┤ X ├────────────■──┤ X ├┤ H ├┤ H ├┤ X ├────────────■──┤ X ├┤ H ├»
«     ├───┤├───┤            │  ├───┤├───┤├───┤├───┤            │  ├───┤├───┤»
«q_1: ┤ H ├┤ X ├────────────■──┤ X ├┤ H ├┤ H ├┤ X ├────────────■──┤ X ├┤ H ├»
«     ├───┤├───┤┌───┐┌───┐┌─┴─┐├───┤├───┤├───┤├───┤┌───┐┌───┐┌─┴─┐├───┤├───┤»
«q_2: ┤ H ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├»
«     └───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘»
«c: 3/══════════════════════════════════════════════════════════════════════»
«                                                                           »
«          ┌─┐      
«q_0: ─────┤M├──────
«          └╥┘┌─┐   
«q_1: ──────╫─┤M├───
«     ┌───┐ ║ └╥┘┌─┐
«q_2: ┤ H ├─╫──╫─┤M├
«     └───┘ ║  ║ └╥┘
«c: 3/══════╩══╩══╩═
«           0  1  2

In [1]:
sampler = Sampler()
result = sampler.run(qc,shots = 8192).result()
counts = result.quasi_dists[0]

print(counts)


import matplotlib.pyplot as plt

# Number of qubits
n = 3

# Sort states 0 → 7
states = sorted(counts.keys())

labels = [format(s, f'0{n}b') for s in states]
values = [counts[s] for s in states]
colors = ['tab:red' if label == '000' else 'tab:blue' for label in labels]

plt.figure(figsize=(8, 4))

plt.bar(labels, values,color = colors)

plt.xlabel("Computational Basis States")
plt.ylabel("Probability")

plt.title("3-Qubit Grover Output Distribution (Target = |000⟩)")

plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)

plt.show()


NameError: name 'Sampler' is not defined

#### NOISE EVAULATION

In [10]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (
    NoiseModel,
    amplitude_damping_error,
    phase_damping_error,
    depolarizing_error,
    ReadoutError
)
from qiskit import transpile
import matplotlib.pyplot as plt

In [11]:
def create_noise_model(noise_type='none', noise_strength=0.1):
    """
    Create a noise model with specified type.
    
    Args:
        noise_type: 'amplitude', 'phase', 'depolarizing', 'readout', 'all', or 'none'
        noise_strength: probability parameter (default 0.1)
    
    Returns:
        NoiseModel object
    """
    noise_model = NoiseModel()
    
    if noise_type in ['amplitude', 'all']:
        amp_error = amplitude_damping_error(noise_strength)
        noise_model.add_all_qubit_quantum_error(amp_error, ['h', 'x'])
    
    if noise_type in ['phase', 'all']:
        phase_error = phase_damping_error(noise_strength)
        noise_model.add_all_qubit_quantum_error(phase_error, ['h', 'x'])
    
    if noise_type in ['depolarizing', 'all']:
        cx_error = depolarizing_error(noise_strength, 2)
        noise_model.add_all_qubit_quantum_error(cx_error, ['cx'])
    
    if noise_type in ['readout', 'all']:
        readout_error = ReadoutError([
            [1 - noise_strength, noise_strength],  # P(meas 0|0), P(1|0)
            [noise_strength, 1 - noise_strength]   # P(meas 0|1), P(1|1)
        ])
        noise_model.add_all_qubit_readout_error(readout_error)
    
    return noise_model


In [12]:
def run_circuit_with_noise(qc, noise_type='none', noise_strength=0.1, shots=4096):
    """
    Run a quantum circuit with specified noise model.
    
    Args:
        qc: QuantumCircuit to execute
        noise_type: Type of noise to apply
        noise_strength: Noise parameter
        shots: Number of shots
    
    Returns:
        Dictionary of probabilities
    """
    noise_model = create_noise_model(noise_type, noise_strength)
    sim = AerSimulator(noise_model=noise_model)
    
    result = sim.run(qc, shots=shots).result()
    counts = result.get_counts()
    
    # Convert to probabilities
    probs = {state: count / shots for state, count in counts.items()}
    
    return probs


In [13]:
def test_all_noise_models(qc, noise_strength=0.1, shots=4096, target_state='111'):
    """
    Test circuit under all noise models independently.
    
    Args:
        qc: QuantumCircuit to test
        noise_strength: Noise parameter
        shots: Number of shots
        target_state: The marked state (for highlighting results)
    
    Returns:
        Dictionary mapping noise_type -> probabilities
    """
    noise_types = ['none', 'amplitude', 'phase', 'depolarizing', 'readout']
    results = {}
    
    print("="*60)
    print(f"Testing {qc.num_qubits}-qubit Grover with target |{target_state}⟩")
    print(f"Noise strength: {noise_strength}, Shots: {shots}")
    print("="*60)
    
    for noise_type in noise_types:
        probs = run_circuit_with_noise(qc, noise_type, noise_strength, shots)
        results[noise_type] = probs
        
        # Print success probability
        success_prob = probs.get(target_state, 0.0)
        print(f"\n{noise_type.upper():15s} | P(|{target_state}⟩) = {success_prob:.4f}")
        
        # Print top 3 states
        top_states = sorted(probs.items(), key=lambda x: x[1], reverse=True)[:3]
        for state, prob in top_states:
            marker = " ← TARGET" if state == target_state else ""
            print(f"  |{state}⟩ : {prob:.4f}{marker}")
    
    return results

In [ ]:
def plot_noise_comparison(results, target_state='111'):
    """
    Plot comparison of all noise models.
    
    Args:
        results: Dictionary from test_all_noise_models
        target_state: The marked state to highlight
    """
    noise_types = list(results.keys())
    n_qubits = len(list(results[noise_types[0]].keys())[0])
    all_states = [format(i, f'0{n_qubits}b') for i in range(2**n_qubits)]
    
    fig, axes = plt.subplots(1, len(noise_types), figsize=(4*len(noise_types), 4))
    
    if len(noise_types) == 1:
        axes = [axes]
    
    for ax, noise_type in zip(axes, noise_types):
        probs = results[noise_type]
        values = [probs.get(state, 0.0) for state in all_states]
        colors = ['tab:red' if state == target_state else 'tab:blue' for state in all_states]
        
        ax.bar(all_states, values, color=colors)
        ax.set_xlabel("Basis States")
        ax.set_ylabel("Probability")
        ax.set_title(f"{noise_type.capitalize()} Noise")
        ax.set_ylim(0, 1)
        ax.grid(axis='y', alpha=0.3)
        
        # Rotate x-labels if many qubits
        if n_qubits > 3:
            ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()


In [15]:
def compare_target_states(qc_dict, noise_type='amplitude', noise_strength=0.1, shots=4096):
    """
    Compare different target states under the same noise model.
    
    Args:
        qc_dict: Dictionary mapping target_state -> QuantumCircuit
        noise_type: Type of noise to test
        noise_strength: Noise parameter
        shots: Number of shots
    
    Returns:
        Dictionary mapping target_state -> success_probability
    """
    print("="*60)
    print(f"Comparing target states under {noise_type.upper()} noise")
    print(f"Noise strength: {noise_strength}, Shots: {shots}")
    print("="*60)
    
    comparison = {}
    
    for target_state, qc in qc_dict.items():
        probs = run_circuit_with_noise(qc, noise_type, noise_strength, shots)
        success_prob = probs.get(target_state, 0.0)
        comparison[target_state] = success_prob
        
        hamming_weight = target_state.count('1')
        print(f"\nTarget: |{target_state}⟩ (w={hamming_weight}) → P = {success_prob:.4f}")
    
    return comparison

In [ ]:
results = test_all_noise_models(qc, noise_strength=0.1, target_state='000')
plot_noise_comparison(results, target_state='000')